Проект выполнен на основе датасета "Marketing & E-Commerce Analytics Dataset", размешенного на [kaggle](https://www.kaggle.com/datasets/geethasagarbonthu/marketing-and-e-commerce-analytics-dataset/data) 

## Импорт библиотек

In [5]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine, text

## Подключение к базе и выгрузка данных

Для удобства работы локально была развернута база данных PostgreSQL, в которую импортированы таблицы из исследуемого датасета. Дальнейшая выгрузка данных будет выполняться с помощью SQL-запросов к этой базе

In [10]:
with open('creds.txt', 'r', encoding='utf-8') as file:
    creds = creds = file.read().splitlines()

# создание соединения с базой и проверка
DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': creds[0],  
    'user': creds[1],
    'password': creds[2]  
}

engine = create_engine(
    f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

# проверка соединения
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), version();"))
    db_name, version = result.fetchone()
    print(f"Connected to: {db_name}")
    print(f"PostgreSQL version: {version[:20]}...")

# выводим список таблиц в базе и количество строк в каждой
tables_df = pd.read_sql_query("""
    SELECT 
        schemaname,
        relname as table_name,
        n_live_tup as estimated_rows
    FROM pg_stat_user_tables 
    WHERE schemaname = 'public';
""", engine)

print(tables_df)

Connected to: marketing
PostgreSQL version: PostgreSQL 17.10 on ...
  schemaname    table_name  estimated_rows
0     public        events         2000000
1     public     campaigns              50
2     public      products            2000
3     public     customers          100000
4     public  transactions          103127
